#  تشخیص افراد ایستاده و نشسته با YOLO و mediapipe

In [2]:
# !pip install ultralytics
# !pip install mediapipe

## Import Libraries

In [1]:
import cv2
import numpy as np
from ultralytics import YOLO
import mediapipe as mp
from collections import deque

## Configuration and Hyperparameters

In [2]:

VIDEO_PATH = 0              # 0 for webcam
CONF_THRESH = 0.6           # Minimum confidence for YOLO
TEMPORAL_WINDOW = 15        # تعداد فریم‌هایی که وضعیت‌های اخیر شخص ذخیره می‌شوند
IOU_THRESH = 0.4            # کمتر از این  ← شخص جدید فرض می‌شود
MIN_CONSECUTIVE_FRAMES = 8  # حداقل تعداد فریم متوالی لازم برای تغییر وضعیت
DETECTING_TIME = 2.0        # زمان نشون دادن وضعیت detecting...

# Sitting thresholds 
KNEE_ANGLE_SIT = 120     # اگر زاویه زانو < 120  ← احتمال نشستن
HIP_KNEE_RATIO_SIT = 1.2 # یا اگر نسبت لگن به زانو کوچک شود  ← نشستن


## Model Initialization

In [3]:

yolo = YOLO("yolov8n.pt")   # مدل YOLOv8 
                            # فقط برای تشخیص person

mp_pose = mp.solutions.pose
pose = mp_pose.Pose(
    static_image_mode=False,
    model_complexity=1,
    enable_segmentation=False,
    min_detection_confidence=0.7,
    min_tracking_confidence=0.7
)
                            # MediaPipe Pose
                            # برای استخراج مفاصل بدن
                            # model_complexity=1 → سریع‌تر
                            # min_tracking_confidence=0.7 → پایدارتر


## Geometric Utility Functions

In [4]:

def angle(a, b, c):
    """Angle at point b"""  # برای محاسبه زاویه زانو
    ba = np.array(a) - np.array(b)
    bc = np.array(c) - np.array(b)
    cosang = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-6)
    return np.degrees(np.arccos(np.clip(cosang, -1, 1)))

def iou(boxA, boxB):        # برای match کردن اشخاص در فریم‌های متوالی
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])
    inter = max(0, xB - xA) * max(0, yB - yA)
    areaA = (boxA[2]-boxA[0]) * (boxA[3]-boxA[1])
    areaB = (boxB[2]-boxB[0]) * (boxB[3]-boxB[1])
    return inter / (areaA + areaB - inter + 1e-6)
    

## Person Tracking and State Management

In [5]:

class Person:
    def __init__(self, pid, box):
        self.id = pid
        self.box = box
        self.states = deque(maxlen=TEMPORAL_WINDOW) # وضعیت‌های اخیر (Sitting / Standing / Unknown)
        self.state_history = deque(maxlen=20)     
        self.current_state = "Unknown"              # وضعیت پایدار فعلی
        self.last_known_state = "Unknown"           # آخرین وضعیت معتبر (برای وقتی pose قطع می‌شود)
        self.consecutive_count = 0
        self.first_seen_time = cv2.getTickCount() / cv2.getTickFrequency()  # زمان اولین مشاهده شخص برای پیاده سازی detecting...
        
    def update(self, box, state):
        self.box = box
        
        # اگر pose نتوانسته تصمیم بگیرد آخرین وضعیت حفظ می شود
        if state == "Unknown":
            if self.last_known_state != "Unknown":
                self.states.append(self.last_known_state)
                self.state_history.append(self.last_known_state)
            else:
                self.states.append(state)
                self.state_history.append(state)
        else:
            self.states.append(state)
            self.state_history.append(state)
            self.last_known_state = state
        
        # برای جلوگیری از چشمک زدن
        self._update_state_with_hysteresis()

    def _update_state_with_hysteresis(self):
        """اگر نمونه‌ها کم باشند  ← تصمیم نمی‌گیریم"""
        if len(self.states) < 4:  # Need minimum samples
            self.current_state = "Unknown" if not self.states else self.states[-1]
            return
            
        # رای اکثریت روی ۵ فریم اخیر
        recent_states = list(self.states)[-5:] if len(self.states) >= 5 else list(self.states)
        most_common = max(set(recent_states), key=recent_states.count)
        
        # Check for state change
        if most_common != self.current_state:
            # Count consecutive frames with new state
            consecutive_new = 0
            for s in reversed(list(self.states)):
                if s == most_common:
                    consecutive_new += 1
                else:
                    break
            
            # تغییر وضعیت فقط اگر ۸ فریم پشت سر هم تکرار شده باشد
            if consecutive_new >= MIN_CONSECUTIVE_FRAMES:
                self.consecutive_count = 0
                self.current_state = most_common
            else:
                self.consecutive_count = consecutive_new
        else:
            self.consecutive_count = 0
            
    def stable_state(self):
        now = cv2.getTickCount() / cv2.getTickFrequency()
        if now - self.first_seen_time < DETECTING_TIME:
            return "Detecting..."
        return self.current_state if self.current_state != "Unknown" else "Detecting..."

    # اگر کمتر از ۲ ثانیه گذشته → Detecting...
    # اگر هنوز Unknown → Detecting...


## Visualization and Rendering

In [10]:
people = {}
next_id = 0
frame_count = 0

# ===================== VIDEO =====================
cap = cv2.VideoCapture(VIDEO_PATH)


while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    frame_count += 1
    
    detections = []
    results = yolo(frame, verbose=False)[0]   # تشخیص افراد با YOLO

    for box in results.boxes:
        if int(box.cls[0]) != 0 or box.conf[0] < CONF_THRESH:  # فقط class = person
            continue
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        detections.append((x1, y1, x2, y2))

    matched = set()

    # Track all current people
    current_people = set(people.keys())
    
    for det in detections:               # پیدا کردن نزدیک‌ترین شخص قبلی 
        best_iou, best_id = 0, None
        for pid, person in people.items():
            score = iou(det, person.box)
            if score > best_iou and score > IOU_THRESH:
                best_iou, best_id = score, pid          # اگر IoU کافی نبود  ← شخص جدید

        # ========== POSE ==========
        x1, y1, x2, y2 = det
        #  کراپ کمی بزرگ‌تر برای پایداری 
        h, w = frame.shape[:2]
        x1_exp = max(0, x1 - 10)
        y1_exp = max(0, y1 - 10)
        x2_exp = min(w, x2 + 10)
        y2_exp = min(h, y2 + 10)
        
        crop = frame[y1_exp:y2_exp, x1_exp:x2_exp]  # کراپ اطراف شخص
        if crop.size == 0:
            continue

        rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
        res = pose.process(rgb)                     # استخراج مفاصل

        state = "Unknown"
        if res.pose_landmarks:
            lm = res.pose_landmarks.landmark

            hip = lm[mp_pose.PoseLandmark.LEFT_HIP]
            knee = lm[mp_pose.PoseLandmark.LEFT_KNEE]
            ankle = lm[mp_pose.PoseLandmark.LEFT_ANKLE]
            shoulder = lm[mp_pose.PoseLandmark.LEFT_SHOULDER]

            # محاسبه ویژگی‌های هندسی
            
            knee_angle = angle(
                [hip.x, hip.y],
                [knee.x, knee.y],
                [ankle.x, ankle.y]
            )

            hip_knee_ratio = abs(hip.y - shoulder.y) / (abs(knee.y - ankle.y) + 1e-6)

            if knee_angle < KNEE_ANGLE_SIT or hip_knee_ratio < HIP_KNEE_RATIO_SIT:
                state = "Sitting"
            else:
                state = "Standing"

        if best_id is None:
            people[next_id] = Person(next_id, det)
            people[next_id].update(det, state)
            next_id += 1
        else:
            people[best_id].update(det, state)
            matched.add(best_id)

    # حذف اشخاص ناپدید شده
    disappeared = current_people - matched
    for pid in disappeared:
        if pid in people:
            # چند فریم فرد را نگه دار که شاید دوباره برگرده
            if frame_count % 10 == 0:  # هر سه فریم چک کن
                del people[pid]

    # ===================== DRAW =====================
    # رسم Bounding Box و Label
    for person in people.values():
        x1, y1, x2, y2 = person.box
        label = f"ID {person.id}: {person.stable_state()}"
        
        # Color coding
        if "Standing" in label:
            color = (0,255,0)     # سبز
            
        elif "Sitting" in label:
            color = (0,255,255)   # زرد
            
        elif "Detecting..." in label:
            color = (121,131,248) # صورتی
            
        else:
            color = (0,0,255)     # قرمز برای عدم تشخیص
            
        cv2.rectangle(frame,(x1,y1),(x2,y2),color,3)
        # Background for text
        (text_width, text_height), baseline = cv2.getTextSize(
            label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2
        )
        cv2.rectangle(frame, 
                     (x1, y1 - text_height - 10), 
                     (x1 + text_width, y1), 
                     color, -1)
        cv2.putText(frame, label, (x1, y1 - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 2)

        # درصد پایداری وضعیت 
        stability = len([s for s in person.states if s == state]) / max(1, len(person.states))
        conf_text = f"Stability: {stability:.1%}"
        cv2.putText(frame, conf_text, (x1, y2 + 20),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.4, color, 1)

    # ===================== COUNT =====================
    sitting_count = sum(1 for p in people.values() if p.stable_state() == "Sitting")
    standing_count = sum(1 for p in people.values() if p.stable_state() == "Standing")
    
    
    # Sitting 
    cv2.putText(
        frame,
        f"Sitting: {sitting_count}",
        (15, 35),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.9,
        (0, 255, 255),
        2
    )
    
    # Standing 
    cv2.putText(
        frame,
        f"Standing: {standing_count}",
        (15, 65),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.9,
        (0, 255, 0),
        2
    )


    cv2.imshow("Sitting/Standing detection", frame)
    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()

print(f"Sitting: {sitting_count}  Standing: {standing_count}")  # شمارش نهایی


Sitting: 0  Standing: 1
